# LeetCode 1654 — Minimum Jumps to Reach Home bằng Python

Link đề: https://leetcode.com/problems/minimum-jumps-to-reach-home/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Breadth-First Search

Skeleton LeetCode:

```python
class Solution:
    def minimumJumps(self, forbidden: List[int], a: int, b: int, x: int) -> int:
        ...
```

Notebook đi từ **liệt kê mọi dãy bước nhảy** (vét cạn) → **BFS trên trạng thái `(vị trí, vừa lùi?)`** với một **cận trên** cho vị trí → giải thích vì sao cận `max(x, max(forbidden)) + a + b` là đủ → các biến thể cài đặt BFS (theo tầng, mảng khoảng cách).

## 1. Mục tiêu học

1. Nhận ra bài **đường đi ngắn nhất trên đồ thị không trọng số** ⇒ **BFS**: mỗi bước nhảy có chi phí 1.
2. Hiểu vì sao **vị trí thôi chưa đủ** làm trạng thái: ràng buộc "không lùi hai lần liền" buộc phải lưu thêm **bước trước có phải lùi không**.
3. Biết cách chặn không gian trạng thái **vô hạn** (con bọ có thể nhảy tiến mãi) bằng một **cận trên** hợp lý, và kiểm chứng cận đó.
4. Cài đặt BFS chuẩn với `deque`, tập/mảng `visited` hai chiều, đếm theo tầng.
5. Đối chiếu với vét cạn trên input nhỏ, và với BFS có cận "rất rộng" trên input ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Nhà của một con bọ ở vị trí `x` trên trục số. Con bọ xuất phát ở vị trí `0` và nhảy theo luật:
- Có thể nhảy **tiến** đúng `a` đơn vị (sang phải).
- Có thể nhảy **lùi** đúng `b` đơn vị (sang trái).
- **Không** được nhảy lùi **hai lần liên tiếp**.
- **Không** được nhảy vào các vị trí bị cấm `forbidden`.

Con bọ **được** nhảy vượt qua nhà, nhưng **không** được nhảy tới vị trí **âm**.

Trả về **số bước nhảy ít nhất** để tới đúng vị trí `x`; nếu không thể, trả về `-1`.

**Ví dụ 1:** `forbidden = [14,4,18,1,15]`, `a = 3`, `b = 15`, `x = 9` → `3` (tiến 3 lần: `0 → 3 → 6 → 9`).

**Ví dụ 2:** `forbidden = [8,3,16,6,12,20]`, `a = 15`, `b = 13`, `x = 11` → `-1`.

**Ví dụ 3:** `forbidden = [1,6,2,14,5,17,4]`, `a = 16`, `b = 9`, `x = 7` → `2` (tiến `0 → 16`, lùi `16 → 7`).

**Ràng buộc:**
- `1 ≤ forbidden.length ≤ 1000`
- `1 ≤ a, b, forbidden[i] ≤ 2000`
- `0 ≤ x ≤ 2000`
- Các phần tử `forbidden` khác nhau; `x` không bị cấm.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [
    ([14, 4, 18, 1, 15], 3, 15, 9, 3),
    ([8, 3, 16, 6, 12, 20], 15, 13, 11, -1),
    ([1, 6, 2, 14, 5, 17, 4], 16, 9, 7, 2),
]

def valid_path(forbidden, a, b, x, moves):
    """Kiểm tra một dãy nước đi ('F' tiến, 'B' lùi) có hợp lệ và về đúng x không."""
    bad = set(forbidden)
    pos, prev = 0, None
    for m in moves:
        if m == 'B' and prev == 'B':
            return False
        pos += a if m == 'F' else -b
        if pos < 0 or pos in bad:
            return False
        prev = m
    return pos == x

print("Ví dụ 1 'FFF':", valid_path(*examples[0][:4], "FFF"))
print("Ví dụ 3 'FB' :", valid_path(*examples[2][:4], "FB"))
print("Ví dụ 3 'FBB':", valid_path([1, 6, 2, 14, 5, 17, 4], 16, 9, 7, "FBB"), "(lùi hai lần liền -> không hợp lệ)")

Ví dụ 1 'FFF': True
Ví dụ 3 'FB' : True
Ví dụ 3 'FBB': False (lùi hai lần liền -> không hợp lệ)


## 3. Kiến thức nền

### 3.1 BFS — tìm kiếm theo chiều rộng

BFS duyệt đồ thị theo **từng lớp khoảng cách**: trước hết mọi đỉnh cách nguồn 1 bước, rồi 2 bước, … Trong đồ thị **không trọng số**, lần đầu BFS chạm tới một đỉnh chính là theo **đường ngắn nhất**. Dùng `collections.deque` để `popleft()` trong `O(1)`.

In [2]:
graph = {0: [1, 2], 1: [3], 2: [3, 4], 3: [5], 4: [5], 5: []}
dist = {0: 0}
q = deque([0])
while q:
    u = q.popleft()
    for v in graph[u]:
        if v not in dist:
            dist[v] = dist[u] + 1
            q.append(v)
print("khoảng cách từ 0:", dist)

khoảng cách từ 0: {0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3}


### 3.2 Trạng thái nhiều thành phần (tuple)

Khi luật đi phụ thuộc vào "lịch sử gần", ta gói thêm thông tin vào **trạng thái**: ở đây `(pos, backed)` với `backed = True` nếu bước vừa rồi là lùi. Tuple băm được nên dùng làm khóa `set`/`dict`.

In [3]:
visited = {(0, False)}
print((0, False) in visited, (0, True) in visited)
state = (16, False)
pos, backed = state
print("unpack:", pos, backed)

True False
unpack: 16 False


### 3.3 `set` cho các vị trí bị cấm

`bad = set(forbidden)` để kiểm `pos in bad` trong `O(1)` thay vì `O(len(forbidden))` với list.

In [4]:
forbidden = [14, 4, 18, 1, 15]
bad = set(forbidden)
print([p for p in range(20) if p in bad])

[1, 4, 14, 15, 18]


### 3.4 Duyệt BFS theo tầng

Mẫu "theo tầng": mỗi vòng ngoài xử lý đúng `len(q)` phần tử (một tầng), sau đó tăng `steps`. Tiện khi chỉ cần **số bước** mà không cần lưu khoảng cách từng đỉnh.

In [5]:
q = deque([0]); seen = {0}; steps = 0
while q:
    print(f"tầng {steps}: {list(q)}")
    for _ in range(len(q)):
        u = q.popleft()
        for v in graph[u]:
            if v not in seen:
                seen.add(v); q.append(v)
    steps += 1

tầng 0: [0]
tầng 1: [1, 2]
tầng 2: [3, 4]
tầng 3: [5]


## 4. Ý tưởng nền tảng / trực giác

Mô hình hóa: mỗi **trạng thái** là `(vị trí, bước vừa rồi có phải lùi không)`. Từ `(p, False)` đi được tới `(p + a, False)` và `(p − b, True)`; từ `(p, True)` chỉ đi được `(p + a, False)`. Bỏ các trạng thái có vị trí âm hoặc bị cấm.

Tính tay ví dụ 3 (`a = 16`, `b = 9`, `x = 7`, cấm `{1, 2, 4, 5, 6, 14, 17}`):
- Tầng 0: `(0, F)`.
- Tầng 1: tiến → `(16, F)`; lùi → `−9` âm, bỏ.
- Tầng 2: từ `(16, F)`: tiến → `(32, F)`; lùi → `(7, T)` — **về nhà!** ⇒ đáp án `2`.

Vấn đề lớn: con bọ có thể nhảy tiến **mãi mãi** ⇒ đồ thị vô hạn. Với ví dụ 2 (đáp án `−1`), BFS không có cận sẽ **không bao giờ dừng**. Cần một **cận trên** `U` cho vị trí: chứng minh rằng nếu tới được `x` thì tới được bằng một đường đi không bao giờ vượt `U`.

In [6]:
def neighbors(state, a, b, bad):
    p, backed = state
    out = []
    if p + a not in bad:
        out.append((p + a, False))
    if not backed and p - b >= 0 and p - b not in bad:
        out.append((p - b, True))
    return out

forbidden, a, b, x, _ = examples[2]
bad = set(forbidden)
layer = [(0, False)]
for step in range(3):
    print(f"tầng {step}: {layer}")
    layer = [nb for st in layer for nb in neighbors(st, a, b, bad)]

tầng 0: [(0, False)]
tầng 1: [(16, False)]
tầng 2: [(32, False), (7, True)]


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi dãy nước đi

**Ý tưởng:** thử mọi dãy nước đi độ dài `L = 0, 1, 2, …, Lmax` bằng DFS (cắt nhánh khi vị trí âm, bị cấm, hoặc lùi hai lần). Dãy ngắn nhất về tới `x` là đáp án. Nếu tới `Lmax` vẫn chưa thấy ⇒ trả `−1` (có thể **sai** nếu đáp án dài hơn `Lmax` — đây là nhược điểm cốt lõi của vét cạn).

In [7]:
def min_jumps_enum(forbidden, a, b, x, max_len=18):
    bad = set(forbidden)
    best = [None]
    def dfs(pos, backed, depth, limit):
        if pos == x:
            return True
        if depth == limit:
            return False
        if pos + a not in bad and dfs(pos + a, False, depth + 1, limit):
            return True
        if not backed and pos - b >= 0 and pos - b not in bad and dfs(pos - b, True, depth + 1, limit):
            return True
        return False
    for L in range(max_len + 1):           # "iterative deepening": tăng dần độ dài cho phép
        if dfs(0, False, 0, L):
            return L
    return -1

for f, a, b, x, exp in examples:
    print(min_jumps_enum(f, a, b, x), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
-1 (kỳ vọng -1 )
2 (kỳ vọng 2 )


**Độ phức tạp:** mỗi độ sâu `L` có tới `2^L` dãy ⇒ `O(2^Lmax)`. Đáp án thật có thể lên tới hàng nghìn bước (ví dụ `a = 2000`, `b = 1999`) ⇒ hoàn toàn không khả thi; ngoài ra **không xác định được khi nào dừng** cho trường hợp `−1`.

**Nút thắt:** (1) cùng một trạng thái `(pos, backed)` bị thăm lại vô số lần qua các dãy khác nhau — cần **visited**; (2) cần **cận trên** để không gian trạng thái hữu hạn.

## 6. Ý tưởng thuật toán / bước đột phá

**BFS trên trạng thái `(pos, backed)`**, mỗi trạng thái thăm **một lần**, đường đi đầu tiên chạm `pos == x` là ngắn nhất.

**Cận trên** `U = max(x, max(forbidden)) + a + b`: chỉ xét vị trí `0 ≤ pos ≤ U`. Trực giác:
- Gọi `M = max(x, max(forbidden))`. Mọi vị trí `> M` đều **không bị cấm** và không phải nhà.
- Khi con bọ đã ở vùng `> M`, không còn ô cấm nào cản đường, nên mọi cách "đi lên cao rồi quay về" có thể được **sắp lại thứ tự** các bước tiến/lùi sao cho vị trí không vượt quá `M + a + b` mà **không đổi số bước** (chỉ cần một bước tiến `a` và một bước lùi `b` làm "vùng đệm" phía trên `M`).

Ta **không** chứng minh đầy đủ ở đây; thay vào đó mục 12 **kiểm chứng thực nghiệm**: BFS với cận `U` cho cùng kết quả với BFS cận "rất rộng" trên hàng nghìn input ngẫu nhiên. Nhiều lời giải phổ biến dùng cận cố định `6000` (≥ `2000 + 2000 + 2000`), cũng an toàn với ràng buộc đề.

**Quy tắc chuyển trạng thái:**
- Từ `(p, ·)`: **nếu** `p + a ≤ U` và không cấm và chưa thăm **thì** thêm `(p + a, False)`.
- Từ `(p, False)`: **nếu** `p − b ≥ 0` và không cấm và chưa thăm **thì** thêm `(p − b, True)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS trạng thái `(pos, backed)` với cận cố định 6000

In [8]:
def min_jumps_bfs_6000(forbidden, a, b, x, LIMIT=6000):
    bad = set(forbidden)
    start = (0, False)
    q = deque([(0, False, 0)])
    seen = {start}
    while q:
        p, backed, d = q.popleft()
        if p == x:
            return d
        nxt = [(p + a, False)]
        if not backed:
            nxt.append((p - b, True))
        for np_, nb in nxt:
            if 0 <= np_ <= LIMIT and np_ not in bad and (np_, nb) not in seen:
                seen.add((np_, nb))
                q.append((np_, nb, d + 1))
    return -1

for f, a, b, x, exp in examples:
    print(min_jumps_bfs_6000(f, a, b, x), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
-1 (kỳ vọng -1 )
2 (kỳ vọng 2 )


**Độ phức tạp:** tối đa `2 · 6001` trạng thái, mỗi trạng thái 2 cạnh ⇒ `O(6000)` — hằng số với ràng buộc đề. Bộ nhớ `O(6000)`.

### 7.2 Cách 3 — BFS với cận chặt `U = max(x, max(forbidden)) + a + b`

Cận phụ thuộc input nên nhỏ hơn khi các số nhỏ; BFS dùng **mảng boolean** `seen[backed][pos]` thay cho `set` (nhanh hơn).

In [9]:
def min_jumps_bfs_tight(forbidden, a, b, x):
    U = max(x, max(forbidden)) + a + b
    bad = [False] * (U + 1)
    for f in forbidden:
        bad[f] = True
    seen = [[False] * (U + 1) for _ in range(2)]   # seen[backed][pos]
    seen[0][0] = True
    q = deque([(0, 0)])
    steps = 0
    while q:
        for _ in range(len(q)):
            p, backed = q.popleft()
            if p == x:
                return steps
            f = p + a
            if f <= U and not bad[f] and not seen[0][f]:
                seen[0][f] = True; q.append((f, 0))
            if not backed:
                g = p - b
                if g >= 0 and not bad[g] and not seen[1][g]:
                    seen[1][g] = True; q.append((g, 1))
        steps += 1
    return -1

for f, a, b, x, exp in examples:
    print(min_jumps_bfs_tight(f, a, b, x), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
-1 (kỳ vọng -1 )
2 (kỳ vọng 2 )


**Độ phức tạp:** `O(U)` thời gian và bộ nhớ với `U ≤ 6000`.

### 7.3 Cách 4 — BFS lưu mảng khoảng cách `dist[backed][pos]`

Thay vì đếm theo tầng, lưu khoảng cách từng trạng thái. Tiện khi muốn **truy vết đường đi** (lưu thêm `parent`) hoặc trả lời nhiều truy vấn `x` cùng lúc (khoảng cách tới **mọi** vị trí).

In [10]:
def bfs_all(forbidden, a, b, U):
    bad = set(forbidden)
    INF = -1
    dist = [[INF] * (U + 1) for _ in range(2)]
    parent = {}
    dist[0][0] = 0
    q = deque([(0, 0)])
    while q:
        p, k = q.popleft()
        cand = [(p + a, 0)] + ([(p - b, 1)] if k == 0 else [])
        for np_, nk in cand:
            if 0 <= np_ <= U and np_ not in bad and dist[nk][np_] == INF:
                dist[nk][np_] = dist[k][p] + 1
                parent[(np_, nk)] = (p, k)
                q.append((np_, nk))
    return dist, parent

def min_jumps_dist(forbidden, a, b, x):
    U = max(x, max(forbidden)) + a + b
    dist, _ = bfs_all(forbidden, a, b, U)
    cands = [d for d in (dist[0][x], dist[1][x]) if d != -1]
    return min(cands) if cands else -1

def path_to_home(forbidden, a, b, x):
    U = max(x, max(forbidden)) + a + b
    dist, parent = bfs_all(forbidden, a, b, U)
    ends = [(dist[k][x], k) for k in (0, 1) if dist[k][x] != -1]
    if not ends:
        return None
    _, k = min(ends)
    st = (x, k); path = [x]
    while st != (0, 0):
        st = parent[st]; path.append(st[0])
    return path[::-1]

for f, a, b, x, exp in examples:
    print(min_jumps_dist(f, a, b, x), "(kỳ vọng", exp, ") | đường đi:", path_to_home(f, a, b, x))

3 (kỳ vọng 3 ) | đường đi: [0, 3, 6, 9]
-1 (kỳ vọng -1 ) | đường đi: None
2 (kỳ vọng 2 ) | đường đi: [0, 16, 7]


**Độ phức tạp:** `O(U)` thời gian và bộ nhớ.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 (`a = 3`, `b = 15`, `x = 9`, cấm `{1, 4, 14, 15, 18}`). `U = max(9, 18) + 3 + 15 = 36`.

| tầng | trạng thái trong hàng đợi | ghi chú |
|---|---|---|
| 0 | (0, F) | |
| 1 | (3, F) | lùi từ 0 ra âm |
| 2 | (6, F) | lùi từ 3 ra âm |
| 3 | (9, F) | gặp `x = 9` ⇒ trả `3` |

In [11]:
def trace_bfs(forbidden, a, b, x):
    U = max(x, max(forbidden)) + a + b
    bad = set(forbidden)
    seen = {(0, False)}
    q = deque([(0, False)])
    steps = 0
    print(f"U = max({x}, {max(forbidden)}) + {a} + {b} = {U}")
    while q:
        print(f"tầng {steps}: {[(p, 'T' if k else 'F') for p, k in q]}")
        for _ in range(len(q)):
            p, backed = q.popleft()
            if p == x:
                print(f"  -> gặp nhà x={x} ở tầng {steps}")
                return steps
            for np_, nk in [(p + a, False)] + ([(p - b, True)] if not backed else []):
                reason = None
                if np_ < 0: reason = "âm"
                elif np_ > U: reason = "vượt U"
                elif np_ in bad: reason = "bị cấm"
                elif (np_, nk) in seen: reason = "đã thăm"
                if reason:
                    print(f"     từ {p}: tới {np_} bị loại ({reason})")
                else:
                    seen.add((np_, nk)); q.append((np_, nk))
        steps += 1
    print("  -> hàng đợi rỗng, không tới được")
    return -1

trace_bfs([14, 4, 18, 1, 15], 3, 15, 9)

U = max(9, 18) + 3 + 15 = 36
tầng 0: [(0, 'F')]
     từ 0: tới -15 bị loại (âm)
tầng 1: [(3, 'F')]
     từ 3: tới -12 bị loại (âm)
tầng 2: [(6, 'F')]
     từ 6: tới -9 bị loại (âm)
tầng 3: [(9, 'F')]
  -> gặp nhà x=9 ở tầng 3


3

In [12]:
trace_bfs([1, 6, 2, 14, 5, 17, 4], 16, 9, 7)

U = max(7, 17) + 16 + 9 = 42
tầng 0: [(0, 'F')]
     từ 0: tới -9 bị loại (âm)
tầng 1: [(16, 'F')]
tầng 2: [(32, 'F'), (7, 'T')]
     từ 32: tới 48 bị loại (vượt U)
  -> gặp nhà x=7 ở tầng 2


2

Trace ví dụ 2 (đáp án `−1`) — để ý BFS **dừng được** nhờ cận `U` (`U = 20 + 15 + 13 = 48`):

In [13]:
trace_bfs([8, 3, 16, 6, 12, 20], 15, 13, 11)

U = max(11, 20) + 15 + 13 = 48
tầng 0: [(0, 'F')]
     từ 0: tới -13 bị loại (âm)
tầng 1: [(15, 'F')]
tầng 2: [(30, 'F'), (2, 'T')]
tầng 3: [(45, 'F'), (17, 'T'), (17, 'F')]
     từ 45: tới 60 bị loại (vượt U)
     từ 17: tới 32 bị loại (đã thăm)
tầng 4: [(32, 'T'), (32, 'F'), (4, 'T')]
     từ 32: tới 47 bị loại (đã thăm)
tầng 5: [(47, 'F'), (19, 'T'), (19, 'F')]
     từ 47: tới 62 bị loại (vượt U)
     từ 19: tới 34 bị loại (đã thăm)
     từ 19: tới 6 bị loại (bị cấm)
tầng 6: [(34, 'T'), (34, 'F')]
     từ 34: tới 49 bị loại (vượt U)
     từ 34: tới 49 bị loại (vượt U)
tầng 7: [(21, 'T')]
tầng 8: [(36, 'F')]
     từ 36: tới 51 bị loại (vượt U)
tầng 9: [(23, 'T')]
tầng 10: [(38, 'F')]
     từ 38: tới 53 bị loại (vượt U)
tầng 11: [(25, 'T')]
tầng 12: [(40, 'F')]
     từ 40: tới 55 bị loại (vượt U)
tầng 13: [(27, 'T')]
tầng 14: [(42, 'F')]
     từ 42: tới 57 bị loại (vượt U)
tầng 15: [(29, 'T')]
tầng 16: [(44, 'F')]
     từ 44: tới 59 bị loại (vượt U)
tầng 17: [(31, 'T')]
tầng 18: [(46,

-1

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minimumJumps(self, forbidden: List[int], a: int, b: int, x: int) -> int:
        U = max(x, max(forbidden)) + a + b
        bad = set(forbidden)
        seen = {(0, False)}
        q = deque([(0, False)])
        steps = 0
        while q:
            for _ in range(len(q)):
                p, backed = q.popleft()
                if p == x:
                    return steps
                f = p + a
                if f <= U and f not in bad and (f, False) not in seen:
                    seen.add((f, False))
                    q.append((f, False))
                if not backed:
                    g = p - b
                    if g >= 0 and g not in bad and (g, True) not in seen:
                        seen.add((g, True))
                        q.append((g, True))
            steps += 1
        return -1

for f, a, b, x, exp in examples:
    assert Solution().minimumJumps(f, a, b, x) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Mô hình đồ thị:** đỉnh là trạng thái `(p, backed)` với `p ≥ 0`, `p ∉ forbidden`; cạnh là một bước nhảy hợp lệ. Một dãy nhảy hợp lệ từ `0` tới `x` ⇔ một đường đi từ `(0, False)` tới `(x, False)` hoặc `(x, True)`. Số bước nhảy = độ dài đường đi.

**Trạng thái phải có `backed`:** cùng vị trí `p`, đến bằng bước lùi thì **không** được lùi tiếp, đến bằng bước tiến thì **được**. Hai tình huống có tập nước đi khác nhau nên không thể gộp — gộp lại sẽ bỏ sót đường (mục 13, lỗi 1).

**BFS cho đường ngắn nhất:** đồ thị không trọng số; BFS lấy đỉnh theo thứ tự khoảng cách không giảm nên lần đầu gặp `p == x` là số bước nhỏ nhất. Mỗi trạng thái đưa vào hàng đợi đúng một lần (đánh dấu `seen` khi **đưa vào**), nên thuật toán dừng.

**Cận `U`:** giới hạn vị trí trong `[0, U]` chỉ đúng nếu tồn tại đường tối ưu không vượt `U`. Đây là kết quả đã biết cho bài này (lập luận: phần đường đi nằm trong vùng `> max(x, max(forbidden))` không gặp ô cấm nên có thể **sắp lại thứ tự** các bước tiến/lùi để giữ đường đi luôn `≤ M + a + b` mà không đổi số bước); ta kiểm chứng thực nghiệm ở mục 12 bằng cách so với cận rất lớn.

## 11. Phân tích Time / Space Complexity

Gọi `U = max(x, max(forbidden)) + a + b ≤ 6000`, `F = len(forbidden)`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê dãy nước đi | `O(2^L)` | `O(L)` |
| 2. BFS cận 6000 | `O(6000 + F)` | `O(6000)` |
| 3. BFS cận `U` (mảng) | `O(U + F)` | `O(U)` |
| 4. BFS mảng khoảng cách | `O(U + F)` | `O(U)` |

Mỗi trạng thái có tối đa 2 cạnh, có `2(U + 1)` trạng thái ⇒ tuyến tính theo `U`.

In [15]:
rnd = random.Random(1654)
cases = []
for _ in range(200):
    f = rnd.sample(range(1, 2001), rnd.randint(1, 1000))
    a, b = rnd.randint(1, 2000), rnd.randint(1, 2000)
    x = rnd.choice([v for v in range(0, 2001) if v not in set(f)][:50] + [rnd.randint(0, 2000)])
    if x in set(f):
        continue
    cases.append((f, a, b, x))
for name, fn in [("bfs_6000", min_jumps_bfs_6000), ("bfs_tight", min_jumps_bfs_tight), ("Solution", Solution().minimumJumps)]:
    t0 = time.perf_counter(); res = [fn(*c) for c in cases]; dt = time.perf_counter() - t0
    print(f"{name:<10}: {len(cases)} bộ cỡ lớn, {sum(r != -1 for r in res)} bộ tới được, {dt * 1000:7.1f} ms")

bfs_6000  : 199 bộ cỡ lớn, 22 bộ tới được,    90.0 ms
bfs_tight : 199 bộ cỡ lớn, 22 bộ tới được,    29.3 ms
Solution  : 199 bộ cỡ lớn, 22 bộ tới được,    41.4 ms


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: `x = 0`, chỉ cần tiến, chỉ tới được bằng cách vượt nhà rồi lùi, phải lùi xen kẽ nhiều lần, `a = b`.
3. Đối chiếu với vét cạn (`min_jumps_enum`) trên input nhỏ có đáp án ngắn.
4. **Kiểm chứng cận `U`:** so BFS cận `U` với BFS cận rất rộng (`20·(a + b + max)`) trên input ngẫu nhiên nhỏ.

In [16]:
def min_jumps_wide(forbidden, a, b, x):
    """Tham chiếu: BFS với cận rất rộng."""
    return min_jumps_bfs_6000(forbidden, a, b, x, LIMIT=20 * (a + b + max(forbidden) + x))

ALL = {
    "bfs_6000": min_jumps_bfs_6000,
    "bfs_tight": min_jumps_bfs_tight,
    "dist": min_jumps_dist,
    "Solution": lambda f, a, b, x: Solution().minimumJumps(f, a, b, x),
}

def check(f, a, b, x, exp=None):
    ref = min_jumps_wide(f, a, b, x)
    for name, fn in ALL.items():
        got = fn(list(f), a, b, x)
        assert got == ref, (name, f, a, b, x, got, ref)
    if exp is not None:
        assert ref == exp, (f, a, b, x, ref, exp)
    if ref != -1:
        p = path_to_home(f, a, b, x)
        moves = ''.join('F' if p[i + 1] > p[i] else 'B' for i in range(len(p) - 1))
        assert len(moves) == ref and valid_path(f, a, b, x, moves)
    return ref

for f, a, b, x, exp in examples:
    check(f, a, b, x, exp)
print("Ví dụ đề: OK")

check([5], 3, 2, 0, 0)          # đã ở nhà
check([5], 3, 2, 6, 2)          # tiến 2 lần
check([3], 5, 4, 1, 2)          # 0 -> 5 -> 1
check([2], 3, 2, 1, 2)          # tự dựng: 0 -> 3 -> 1
check([5], 10, 3, 4, -1)        # tự dựng: 10m - 3k = 4 cần k > m (lùi liền) -> không thể
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


Hai edge case cuối là **ví dụ tự dựng** — kỳ vọng ghi trong code được xác nhận bằng BFS cận rộng (đối chiếu lại đề gốc nếu muốn). Hãy xem đường đi cụ thể của vài edge case:

In [17]:
for f, a, b, x in [([2], 3, 2, 1), ([3], 5, 4, 1), ([1, 2, 3], 4, 3, 5), ([7], 5, 3, 4)]:
    print(f"forbidden={f}, a={a}, b={b}, x={x}: số bước = {Solution().minimumJumps(f, a, b, x)}, đường đi = {path_to_home(f, a, b, x)}")

forbidden=[2], a=3, b=2, x=1: số bước = 2, đường đi = [0, 3, 1]
forbidden=[3], a=5, b=4, x=1: số bước = 2, đường đi = [0, 5, 1]
forbidden=[1, 2, 3], a=4, b=3, x=5: số bước = 3, đường đi = [0, 4, 8, 5]
forbidden=[7], a=5, b=3, x=4: số bước = -1, đường đi = None


In [18]:
rnd = random.Random(2019)
agree_enum = 0
for _ in range(1500):
    a, b = rnd.randint(1, 12), rnd.randint(1, 12)
    f = rnd.sample(range(1, 30), rnd.randint(1, 10))
    x = rnd.randint(0, 25)
    if x in f:
        continue
    r = check(f, a, b, x)
    if r != -1 and r <= 12:
        assert min_jumps_enum(f, a, b, x, max_len=12) == r
        agree_enum += 1
print(f"1500 bộ ngẫu nhiên nhỏ: cận U khớp cận rộng; {agree_enum} bộ có đáp án ngắn khớp với vét cạn: OK")

for _ in range(15):
    a, b = rnd.randint(1, 2000), rnd.randint(1, 2000)
    f = rnd.sample(range(1, 2001), rnd.randint(1, 1000))
    x = rnd.randint(0, 2000)
    if x in f:
        continue
    assert min_jumps_bfs_tight(f, a, b, x) == min_jumps_bfs_6000(f, a, b, x)
print("15 bộ cỡ đề: cận U khớp cận 6000: OK")

1500 bộ ngẫu nhiên nhỏ: cận U khớp cận rộng; 396 bộ có đáp án ngắn khớp với vét cạn: OK
15 bộ cỡ đề: cận U khớp cận 6000: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đánh dấu `visited` chỉ theo **vị trí**

In [19]:
def min_jumps_bug_pos_only(forbidden, a, b, x):
    U = max(x, max(forbidden)) + a + b
    bad = set(forbidden)
    seen = {0}                                   # SAI: chỉ lưu vị trí
    q = deque([(0, False, 0)])
    while q:
        p, backed, d = q.popleft()
        if p == x:
            return d
        for np_, nb in [(p + a, False)] + ([(p - b, True)] if not backed else []):
            if 0 <= np_ <= U and np_ not in bad and np_ not in seen:
                seen.add(np_); q.append((np_, nb, d + 1))
    return -1

rnd = random.Random(11)
found = None
while found is None:
    a, b = rnd.randint(1, 10), rnd.randint(1, 10)
    f = rnd.sample(range(1, 25), rnd.randint(1, 8))
    x = rnd.randint(0, 20)
    if x not in f and min_jumps_bug_pos_only(f, a, b, x) != Solution().minimumJumps(f, a, b, x):
        found = (f, a, b, x)
f, a, b, x = found
print(f"phản ví dụ: forbidden={f}, a={a}, b={b}, x={x}")
print("   chỉ lưu vị trí:", min_jumps_bug_pos_only(f, a, b, x), "| đúng:", Solution().minimumJumps(f, a, b, x), "| đường đi đúng:", path_to_home(f, a, b, x))

phản ví dụ: forbidden=[3, 18, 23], a=8, b=5, x=20
   chỉ lưu vị trí: -1 | đúng: 9 | đường đi đúng: [0, 8, 16, 11, 19, 14, 22, 17, 25, 20]


**Vì sao sai:** lần đầu tới vị trí `p` có thể là bằng **bước lùi** (nên không lùi tiếp được), trong khi đường tối ưu cần đến `p` bằng **bước tiến** rồi lùi. Đánh dấu chỉ theo vị trí chặn mất trạng thái thứ hai. **Sửa:** `seen` chứa `(pos, backed)`.

### Lỗi 2 — Không cho nhảy vượt quá nhà (cận trên `= x`)

In [20]:
def min_jumps_bug_no_overshoot(forbidden, a, b, x):
    return min_jumps_bfs_6000(forbidden, a, b, x, LIMIT=x)      # SAI: cận quá nhỏ

f, a, b, x, exp = examples[2]
print("cận = x:", min_jumps_bug_no_overshoot(f, a, b, x), "| đúng:", Solution().minimumJumps(f, a, b, x))

cận = x: -1 | đúng: 2


**Vì sao sai:** ví dụ 3 bắt buộc phải lên `16` rồi lùi về `7`. Đề nói rõ con bọ được vượt qua nhà. **Sửa:** dùng cận `max(x, max(forbidden)) + a + b` (hoặc 6000).

### Lỗi 3 — Cho phép lùi hai lần liên tiếp

In [21]:
def min_jumps_bug_double_back(forbidden, a, b, x):
    U = max(x, max(forbidden)) + a + b
    bad = set(forbidden)
    seen = {0}
    q = deque([(0, 0)])
    while q:
        p, d = q.popleft()
        if p == x:
            return d
        for np_ in (p + a, p - b):                 # SAI: bỏ qua luật "không lùi hai lần liền"
            if 0 <= np_ <= U and np_ not in bad and np_ not in seen:
                seen.add(np_); q.append((np_, d + 1))
    return -1

f, a, b, x = [5], 10, 3, 4
print("cho lùi liên tiếp:", min_jumps_bug_double_back(f, a, b, x), "| đúng:", Solution().minimumJumps(f, a, b, x))
print("đường đi đúng:", path_to_home(f, a, b, x))

cho lùi liên tiếp: 3 | đúng: -1
đường đi đúng: None


**Vì sao sai:** bản sai đi `0 → 10 → 7 → 4` (lùi hai lần liền, 3 bước), vi phạm luật. Với luật đúng, muốn tới `4` cần `10m − 3k = 4` với số lần lùi `k ≤ m` (không lùi liền) — vô nghiệm, nên đáp án đúng là `−1`. **Sửa:** chỉ cho lùi khi `backed == False`.

### Lỗi 4 — Kiểm tra `p == x` khi **lấy ra** nhưng đánh dấu `seen` khi **lấy ra** (thay vì khi đưa vào)

Đánh dấu muộn khiến một trạng thái bị đưa vào hàng đợi **nhiều lần** — kết quả vẫn đúng nhưng hàng đợi phình to. Ta đo số phần tử từng được đưa vào:

In [22]:
def bfs_count_pushes(forbidden, a, b, x, mark_on_push=True):
    U = max(x, max(forbidden)) + a + b
    bad = set(forbidden)
    seen = set()
    if mark_on_push:
        seen.add((0, False))
    q = deque([(0, False, 0)]); pushes = 1
    while q:
        p, k, d = q.popleft()
        if not mark_on_push:
            if (p, k) in seen:
                continue
            seen.add((p, k))
        if p == x:
            return d, pushes
        for np_, nk in [(p + a, False)] + ([(p - b, True)] if not k else []):
            if 0 <= np_ <= U and np_ not in bad and (np_, nk) not in seen:
                if mark_on_push:
                    seen.add((np_, nk))
                q.append((np_, nk, d + 1)); pushes += 1
    return -1, pushes

f, a, b, x = [2000], 7, 5, 1999
print("đánh dấu khi đưa vào :", bfs_count_pushes(f, a, b, x, True))
print("đánh dấu khi lấy ra  :", bfs_count_pushes(f, a, b, x, False))

đánh dấu khi đưa vào : (289, 3954)
đánh dấu khi lấy ra  : (289, 5640)


**Sửa:** đánh dấu `seen` **ngay khi đưa vào hàng đợi** — mỗi trạng thái vào hàng đợi đúng một lần.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số bước **ít nhất**", mỗi bước chi phí như nhau ⇒ **BFS**.
- Luật đi phụ thuộc bước trước ("không lùi hai lần liền", "không rẽ trái hai lần"…) ⇒ thêm thông tin đó vào **trạng thái**.
- Không gian vô hạn (được đi xa tùy ý) ⇒ tìm **cận trên**; nếu không chứng minh được cận chặt, dùng cận rộng an toàn theo ràng buộc (ở đây 6000) và kiểm thực nghiệm.
- Đánh dấu `visited` **khi đưa vào** hàng đợi.
- Muốn in đường đi: lưu `parent[state]`.

## 15. Biến thể của bài

1. **Không giới hạn lùi liên tiếp:** trạng thái chỉ cần vị trí.
2. **Không được lùi quá `k` lần liên tiếp:** trạng thái `(pos, số lần lùi liên tiếp hiện tại)`, `k + 1` lớp.
3. **Mỗi bước có chi phí khác nhau** (tiến tốn `ca`, lùi tốn `cb`): dùng **Dijkstra** hoặc 0-1 BFS nếu chi phí 0/1.
4. **LeetCode 1306 — Jump Game III:** nhảy `±arr[i]`, hỏi có tới được ô giá trị 0 không — BFS/DFS trên chỉ số.
5. **LeetCode 1345 — Jump Game IV:** BFS với cạnh "cùng giá trị".
6. **Trả về mọi vị trí tới được trong `≤ k` bước:** BFS theo tầng dừng ở tầng `k`.

In [23]:
def min_jumps_k_back(forbidden, a, b, x, k):
    """Biến thể 2: được lùi tối đa k lần liên tiếp. k = 1 là bài gốc."""
    U = max(x, max(forbidden)) + a + (k + 1) * b
    bad = set(forbidden)
    seen = {(0, 0)}
    q = deque([(0, 0, 0)])
    while q:
        p, c, d = q.popleft()
        if p == x:
            return d
        cand = [(p + a, 0)] + ([(p - b, c + 1)] if c < k else [])
        for np_, nc in cand:
            if 0 <= np_ <= U * 3 and np_ not in bad and (np_, nc) not in seen:
                seen.add((np_, nc)); q.append((np_, nc, d + 1))
    return -1

rnd = random.Random(15)
for _ in range(500):
    a, b = rnd.randint(1, 10), rnd.randint(1, 10)
    f = rnd.sample(range(1, 25), rnd.randint(1, 8)); x = rnd.randint(0, 20)
    if x in f: continue
    assert min_jumps_k_back(f, a, b, x, 1) == Solution().minimumJumps(f, a, b, x)
print("k=1 khớp bài gốc. Ví dụ [5], a=10, b=3, x=4:", [min_jumps_k_back([5], 10, 3, 4, k) for k in (1, 2, 3)])

k=1 khớp bài gốc. Ví dụ [5], a=10, b=3, x=4: [-1, 3, 3]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê dãy | iterative deepening DFS | `O(2^L)` | `O(L)` | kiểm thử đáp án ngắn |
| 2. BFS cận 6000 | trạng thái `(pos, backed)`, cận cố định | `O(6000)` | `O(6000)` | an toàn, dễ nhớ |
| 3. BFS cận `U` | cận phụ thuộc input, mảng boolean | `O(U)` | `O(U)` | **nộp bài** (bản `set` ở mục 9) |
| 4. BFS `dist` + `parent` | khoảng cách tới mọi trạng thái | `O(U)` | `O(U)` | cần đường đi / nhiều truy vấn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimumJumps([10], 4, 3, 5)` bằng bao nhiêu? Đường đi là gì?

**Bài 2 (tính tay):** Viết các tầng BFS cho `forbidden = [3]`, `a = 5`, `b = 4`, `x = 2`, tới khi gặp nhà.

**Bài 3 (điền TODO):** Hoàn thiện `reachable_positions(forbidden, a, b, U)` trả về **tập mọi vị trí** trong `[0, U]` mà con bọ tới được.

**Bài 4 (tự viết lại):** Viết lại BFS mà **không** dùng đếm tầng — lưu `d` trong hàng đợi — và in thêm đường đi bằng `parent`.

**Bài 5 (mini challenge):** Biến thể chi phí: bước tiến tốn `ca`, bước lùi tốn `cb`. Viết `min_cost_home(forbidden, a, b, x, ca, cb)` bằng Dijkstra trên trạng thái `(pos, backed)` (cận `U` như cũ), kiểm rằng `ca = cb = 1` khớp với `Solution`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def reachable_positions(forbidden, a, b, U):
    # TODO: BFS từ (0, False); gom mọi pos đã thăm (cả hai trạng thái backed)
    pass

# Bài 4 — TODO
def min_jumps_with_path(forbidden, a, b, x):
    # TODO: hàng đợi chứa (pos, backed, d); parent[(np, nb)] = (p, backed)
    pass

# Bài 5 — TODO
def min_cost_home(forbidden, a, b, x, ca, cb):
    # TODO: heapq với (cost, pos, backed); dist dict; bỏ qua phần tử cũ
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Nhà `5` không chia hết cho `4`.
- Mức 2: Cần tiến nhiều lần rồi lùi: `4k − 3m = 5`, lùi không liền nhau.
- Mức 3: Thử `0 → 4 → 8 → 5`.

**Bài 2**
- Mức 1: Tầng 1: `(5, F)`.
- Mức 2: Tầng 2: `(10, F)`, `(1, T)`.
- Mức 3: Tầng 3: `(15, F)`, `(6, T)` (từ 10), `(6, F)` (từ 1)… tiếp tục tới khi thấy `2`.

**Bài 3**
- Mức 1: Giống `Solution` nhưng không dừng khi gặp `x`.
- Mức 2: Kết quả là `{p for (p, k) in seen}`.
- Mức 3: Kiểm: mọi `x` trong tập có `Solution(...) != −1`.

**Bài 4**
- Mức 1: Lưu `parent` khi đưa vào hàng đợi.
- Mức 2: Đi ngược từ `(x, k)` về `(0, False)`.
- Mức 3: Kiểm độ dài đường đi − 1 = số bước, và `valid_path` đúng.

**Bài 5**
- Mức 1: Chi phí khác nhau ⇒ BFS không còn đúng ⇒ Dijkstra.
- Mức 2: Trạng thái giống hệt, chỉ khác hàng đợi ưu tiên.
- Mức 3: Kết quả là `min(dist[(x, F)], dist[(x, T)])`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
print("Bài 1:", Solution().minimumJumps([10], 4, 3, 5), "| đường đi:", path_to_home([10], 4, 3, 5))
assert Solution().minimumJumps([10], 4, 3, 5) == 3

# Bài 2
ans2 = trace_bfs([3], 5, 4, 2)
assert ans2 == min_jumps_wide([3], 5, 4, 2)

Bài 1: 3 | đường đi: [0, 4, 1, 5]
U = max(2, 3) + 5 + 4 = 12
tầng 0: [(0, 'F')]
     từ 0: tới -4 bị loại (âm)
tầng 1: [(5, 'F')]
tầng 2: [(10, 'F'), (1, 'T')]
     từ 10: tới 15 bị loại (vượt U)
tầng 3: [(6, 'T'), (6, 'F')]
     từ 6: tới 11 bị loại (đã thăm)
tầng 4: [(11, 'F'), (2, 'T')]
     từ 11: tới 16 bị loại (vượt U)
  -> gặp nhà x=2 ở tầng 4


In [26]:
# Bài 3
def reachable_positions_sol(forbidden, a, b, U):
    bad = set(forbidden)
    seen = {(0, False)}
    q = deque([(0, False)])
    while q:
        p, k = q.popleft()
        for np_, nk in [(p + a, False)] + ([(p - b, True)] if not k else []):
            if 0 <= np_ <= U and np_ not in bad and (np_, nk) not in seen:
                seen.add((np_, nk)); q.append((np_, nk))
    return {p for p, _ in seen}

f, a, b = [14, 4, 18, 1, 15], 3, 15
U = max(max(f), 40) + a + b
R = reachable_positions_sol(f, a, b, U)
print("Bài 3: vị trí tới được trong [0, 40]:", sorted(p for p in R if p <= 40))
for xx in range(0, 41):
    if xx not in f:
        assert (xx in R) == (Solution().minimumJumps(f, a, b, xx) != -1)

# Bài 4
def min_jumps_with_path_sol(forbidden, a, b, x):
    U = max(x, max(forbidden)) + a + b
    bad = set(forbidden)
    parent = {(0, False): None}
    q = deque([(0, False, 0)])
    while q:
        p, k, d = q.popleft()
        if p == x:
            path, st = [], (p, k)
            while st is not None:
                path.append(st[0]); st = parent[st]
            return d, path[::-1]
        for np_, nk in [(p + a, False)] + ([(p - b, True)] if not k else []):
            if 0 <= np_ <= U and np_ not in bad and (np_, nk) not in parent:
                parent[(np_, nk)] = (p, k); q.append((np_, nk, d + 1))
    return -1, None

rnd = random.Random(44)
for _ in range(800):
    a, b = rnd.randint(1, 12), rnd.randint(1, 12)
    f = rnd.sample(range(1, 30), rnd.randint(1, 10)); x = rnd.randint(0, 25)
    if x in f: continue
    d, path = min_jumps_with_path_sol(f, a, b, x)
    assert d == Solution().minimumJumps(f, a, b, x)
    if d != -1:
        moves = ''.join('F' if path[i + 1] > path[i] else 'B' for i in range(len(path) - 1))
        assert len(moves) == d and valid_path(f, a, b, x, moves)
print("Bài 4 OK | ví dụ 3:", min_jumps_with_path_sol([1, 6, 2, 14, 5, 17, 4], 16, 9, 7))

Bài 3: vị trí tới được trong [0, 40]: [0, 3, 6, 9, 12]
Bài 4 OK | ví dụ 3: (2, [0, 16, 7])


In [27]:
# Bài 5
import heapq

def min_cost_home_sol(forbidden, a, b, x, ca, cb):
    U = max(x, max(forbidden)) + a + b
    bad = set(forbidden)
    dist = {(0, False): 0}
    pq = [(0, 0, False)]
    while pq:
        c, p, k = heapq.heappop(pq)
        if c > dist.get((p, k), float('inf')):
            continue
        if p == x:
            return c
        for np_, nk, w in [(p + a, False, ca)] + ([(p - b, True, cb)] if not k else []):
            if 0 <= np_ <= U and np_ not in bad:
                nc = c + w
                if nc < dist.get((np_, nk), float('inf')):
                    dist[(np_, nk)] = nc
                    heapq.heappush(pq, (nc, np_, nk))
    return -1

rnd = random.Random(55)
for _ in range(800):
    a, b = rnd.randint(1, 12), rnd.randint(1, 12)
    f = rnd.sample(range(1, 30), rnd.randint(1, 10)); x = rnd.randint(0, 25)
    if x in f: continue
    assert min_cost_home_sol(f, a, b, x, 1, 1) == Solution().minimumJumps(f, a, b, x)
print("Bài 5 OK | ví dụ 3 với tiến tốn 5, lùi tốn 1:", min_cost_home_sol([1, 6, 2, 14, 5, 17, 4], 16, 9, 7, 5, 1))

Bài 5 OK | ví dụ 3 với tiến tốn 5, lùi tốn 1: 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS trên trạng thái có "trí nhớ" + cận trên:**

```python
U = max(x, max(forbidden)) + a + b          # cận trên cho vị trí
bad = set(forbidden)
seen = {(0, False)}
q = deque([(0, False)])
steps = 0
while q:
    for _ in range(len(q)):
        p, backed = q.popleft()
        if p == x:
            return steps
        for np, nb in [(p + a, False)] + ([(p - b, True)] if not backed else []):
            if 0 <= np <= U and np not in bad and (np, nb) not in seen:
                seen.add((np, nb))          # đánh dấu khi ĐƯA VÀO
                q.append((np, nb))
    steps += 1
return -1
```

**Checklist:**
- [ ] Trạng thái gồm `(pos, backed)`, không chỉ `pos`?
- [ ] Không lùi khi `backed == True`?
- [ ] Không vào vị trí âm hay bị cấm?
- [ ] Có cận trên (cho phép vượt nhà) để BFS dừng khi đáp án là `−1`?
- [ ] `x = 0` trả `0` ngay?
- [ ] Đánh dấu `seen` khi đưa vào hàng đợi?
- [ ] Chữ ký đúng: `def minimumJumps(self, forbidden: List[int], a: int, b: int, x: int) -> int`?